In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sys
from tqdm import tqdm

PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT)) 

from roi_classifier.prepare_data import prepare_roi_data
from roi_classifier.annotate_data import annotate_rois
from roi_classifier.train_classifier import train_roi_classifier




In [2]:
DATASET_ROOT = Path(r"C:\Users\mzinn1\Desktop\invivo_tiffs")  # TODO: set this to your data path
assert DATASET_ROOT.exists(), f"Dataset root {DATASET_ROOT} does not exist."

ROI_DIR = PROJECT_ROOT / "data"
ROI_DIR.mkdir(parents=True, exist_ok=True)

ROI_DATA_PATH = ROI_DIR / "invivo_roi_features.npy"

MODEL_OUT_DIR = PROJECT_ROOT / "models"
MODEL_OUT_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = PROJECT_ROOT / "config/classifier_config.yaml"

print(f"Extracting fluorescence data from {DATASET_ROOT.__str__()}")
print(f"Saving engineered data to {ROI_DATA_PATH.__str__()}")
print(f"Saving models to {MODEL_OUT_DIR.__str__()}")
print(f"Configuring classifier according to {CONFIG_PATH.__str__()}")

Extracting fluorescence data from C:\Users\mzinn1\Desktop\invivo_tiffs
Saving engineered data to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features.npy
Saving models to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\models
Configuring classifier according to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\config\classifier_config.yaml


In [3]:
update = True # Change to false 
backup = False # Change as you wish; controls whether or not a backup of the original engineered data is saved
fs = 3.0       # Frame rate in Hz — set to match your acquisition rate (scales smoothing accordingly)

roi_data = prepare_roi_data(
    dataset_root=DATASET_ROOT,
    input_file=ROI_DATA_PATH,
    output_file=ROI_DATA_PATH,
    update=update,
    backup=backup,
    fs=fs
)



  ROI Summary
  Total rois: 1942
  Good: 144 | Bad: 357 | Unlabeled: 1441
  Manual: 500 | Auto: 1
  Total spikes stored: 5301

  Re-smoothing with sigma=0.80 (fs=3.0, base sigma=4.0)

Updated 1942 ROIs
  - Re-smoothed 1942 ROIs from raw traces
  - Preserved 500 manual labels
  - Preserved 5301 spikes

Saved 1942 ROIs to C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\data\invivo_roi_features.npy


In [7]:
# Change these flags to control which ROIs are shown for annotation and how many
unlabeled_only = False 
labeled_only = True
n_samples = 1000

assert not (unlabeled_only and labeled_only), "unlabeled_only and labeled_only cannot both be True — pick one or set both to False to show all ROIs."

annotate_rois(data_path=ROI_DATA_PATH,
              n_samples=n_samples,
              unlabeled_only=unlabeled_only,
              labeled_only=labeled_only)

Found 501 roi keys matching filter
Returning all 501 keys.
[1/501] Skipped: 5729L-13_27
[2/501] Skipped: 5729L-21_62
[3/501] Skipped: 5729L-27_0
[4/501] Skipped: 5735R-13_45
[5/501] Skipped: 5729L-13_44
[6/501] Skipped: 5730R-4_54
[7/501] Skipped: 5730R-7_12
[8/501] Skipped: 5730R-16_48
[9/501] Skipped: 5730R-14_25
[10/501] Skipped: 5735R-13_13
[11/501] Skipped: 5735R-15_15
[12/501] Skipped: 5735L-4_6
[13/501] Skipped: 5730R-15_21
[14/501] Skipped: 5729L-12_27
[15/501] Skipped: 5732R-6_19
[16/501] Skipped: 5729L-26_9
[17/501] Skipped: 5730R-5_23
[18/501] Skipped: 5732L-7_42
[19/501] Skipped: 5735R-10_5
[20/501] Skipped: 5730R-7_3
[21/501] Skipped: 5729L-8_11
[22/501] Skipped: 5730R-4_79
[23/501] Skipped: 5735R-12_34
[24/501] Skipped: 5729L-27_20
[25/501] Skipped: 5732R-7_17
[26/501] Skipped: 5730R-5_33
[27/501] Skipped: 5735R-4_16
[28/501] Skipped: 5732L-2_1
[29/501] Skipped: 5729R-5_19
[30/501] Skipped: 5729L-21_18
[31/501] Skipped: 5730R-15_16
[32/501] Skipped: 5729R-5_13
[33/501] Sk

{'level': 'roi',
 'queued': 501,
 'total': 508,
 'labeled': 0,
 'updated': 0,
 'confirmed': 0,
 'skipped': 508}

In [6]:
name = "invivo_roi_classifier" # TODO Change this as needed for your own experimental/organizational needs
data_paths = [ROI_DATA_PATH] # Can be a list of paths if you have engineered data from multiple sources you want to combine for training
results = train_roi_classifier(config_path=CONFIG_PATH, data_path=data_paths, name=name,
                     output_dir=MODEL_OUT_DIR, verbose=True, manual_only=True, overwrite=False)

Dataset Summary
--------------------------------------------------
Total labeled datapoints: 500
  Train: 400 | Test: 100

Label distribution:
              Bad (0)  Good (1)
  Train           284       116
  Test             72        28
  Total           356       144

Training on: Manual labels only

--------------------------------------------------
TUNED MODEL SUMMARY
--------------------------------------------------
Model:     RandomForestClassifier
Transform: sqrt
Features:  ['var_of_var', 'derivative_skew', 'spike_prom_skew', 'peak_density', 'ac_decay', 'range_trace', 'spike_prom_mean', 'derivative_asymmetry', 'snr_estimate', 'median_spike_prom']

Hyperparameters:
  class_weight: None
  max_depth: None
  min_samples_leaf: 1
  min_samples_split: 10
  n_estimators: 200

Metrics:
  CV Accuracy:   0.8900
  Test Accuracy: 0.9400
  ROC AUC:       0.9896
  F1:            0.9393
  Precision:     0.9395
  Recall:        0.9400

Confusion Matrix:
              Pred 0  Pred 1
  Actual 0 